In [5]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [6]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = make_moons(
    n_samples=1200,
    noise=0.25,
    random_state=42
)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [8]:
def build_model(initializer):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),

        tf.keras.layers.Dense(32, activation="relu", kernel_initializer=initializer),

        tf.keras.layers.Dense(16, activation="relu", kernel_initializer=initializer),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=['accuracy']
    )

    return model

In [9]:
initializers = {
    "Glorot Normal" : tf.keras.initializers.GlorotNormal,
    "He Normal" : tf.keras.initializers.HeNormal,
    "Random Normal" : tf.keras.initializers.RandomNormal
}

results = {}

for name, initializer_class in initializers.items():
    print(f"\nTraining with {name}")
    
    tf.keras.utils.set_random_seed(42)
    model=build_model(initializer_class())
    first_layer_weights = model.layers[0].get_weights()[0]

    print("Initial weight mean : ", round(float(np.mean(first_layer_weights)), 4))
    print("Initial wight standard deviation : ", round(float(np.std(first_layer_weights)), 4))

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    results[name] = {
        "test_loss" : test_loss,
        "test_accuracy" : test_accuracy
    }

print("\nINITIALIZER COMPARISION")
for name, metrics in results.items():
    print(f"\n{name}")
    for metric_name, value in metrics.items():
        print(f"{metric_name} : {value:.4f}")


Training with Glorot Normal
Initial weight mean :  0.0163
Initial wight standard deviation :  0.2753

Training with He Normal
Initial weight mean :  0.0671
Initial wight standard deviation :  1.135

Training with Random Normal
Initial weight mean :  0.0006
Initial wight standard deviation :  0.0557

INITIALIZER COMPARISION

Glorot Normal
test_loss : 0.1482
test_accuracy : 0.9417

He Normal
test_loss : 0.1665
test_accuracy : 0.9375

Random Normal
test_loss : 0.2884
test_accuracy : 0.8917
